<a href="https://colab.research.google.com/github/DavidSenseman/STA1403/blob/main/STA1403_Chapter_10B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- STA1403_CHAPTER_10B:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **STA 1403: Biostatistics**

## **Analysis of Categorical Variables**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

#### Contents
* 10.1 : Introduction**
* 10.2 : Pearson’s χ2 Test for One Categorical Variable
* 10.3 : Pearson’s χ2 Test of Independence
* **10.4 : Entering Contingency Tables**
* **10.5 : Advanced**

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which **_is_** your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your STA1403 Key

Run the next cell to test whether your STA1403 Key is correctly installed in your Colab Secrets.

In [ ]:
# @title Test Your STA1403_KEY

from google.colab import userdata
import os

# Check if STA1403 key is properly loaded
try:
    # 1. Get the key from Secrets
    STA1403_KEY = userdata.get('STA1403_KEY')

    # 2. Set it as an environment variable
    os.environ['STA1403_KEY'] = STA1403_KEY

    print("STA1403 key loaded and environment variable set successfully!")
    print(f"Key length: {len(STA1403_KEY)}")

except Exception as e:
    print(f"Error loading STA1403 Key: {e}")
    print("Please set your STA1403 Key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'STA1403_KEY'")
    print("3. Paste your STA1403 key and toggle 'Notebook access' on")

If your STA1403 API Key is correctly stored in your Colab Secrets, you should see the following output:
```text
STA1403 key loaded and environment variable set successfully!
Key length: 28
```
Contact your Instructor if you recieve an error since you won't be able to submit your lesson for grading unless your API is correctly installed.

## Load Data Sets

Run the next cell to load the data sets for this lesson.

In [ ]:
# @title Load Data Sets
import pandas as pd
import seaborn as sns

# Load Titanic dataset and subset to two columns
df_titanic = sns.load_dataset('titanic')[['sex', 'survived']].dropna()
df_titanic.columns = ['Sex', 'Survived']
df_titanic['Survived'] = df_titanic['Survived'].map({1: 'Yes', 0: 'No'})

# Load dataset
# UCB Admissions dataset (pre-aggregated frequency format)
df_ucb = pd.DataFrame({
    'Admit':  ['Admitted']*6 + ['Rejected']*6,
    'Dept':   ['A','B','C','D','E','F'] * 2,
    'Freq':   [601, 370, 322, 269, 147, 46,
               332, 138, 347, 579, 480, 668]
})

# Load dataset
df_aspirin = pd.read_csv("https://biologicslab.co/STA1403/data/aspirin.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load dataset
df_bwt = pd.read_csv("https://biologicslab.co/STA1403/data/birthwt",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load dataset
df_hair_eye = pd.read_csv("https://biologicslab.co/STA1403/data/haireyecolor.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load dataset
df_pima = pd.read_csv("https://biologicslab.co/STA1403/data/Pima.tr",
                      sep=',', na_values=[" ", "NA", "null", ""])
df_pima["bmi"] = df_pima["bmi"].fillna(df_pima["bmi"].median())

print("Data sets loaded sucessfully. ✅")

# **Chapter 10 : Analysis of Categorical Variables**

## **10.4 Entering Contingency Tables into Python**

In Python, we can enter and analyze contingency tables without importing individual observations. For example, let us consider the study for investigating whether regular aspirin intake reduces the mortality from cardiovascular disease [36]. In this case, the null hypothesis is that the heart attack is independent of aspirin intake. Based on the contingency table (Table 10.7) of the observed frequencies, we can evaluate the strength of relationship between these two binary variables.

![__](https://biologicslab.co/STA1403/images/chapter_10/chapter_10A_image14A.png)

>**Table 10.7** Contingency table of heart attack status by the type of treatment



## Example 1: Generate and Analyze Contingency Table

The code in the cell below show how to generate and analyze a contingency table for the `df_aspirin` DataFrame. The code tests whether there is a statistically significant association between aspirin treatment and heart attack status.

In [ ]:
# @title Example 1: Generate and Analyze Contingency Table

import pandas as pd
import numpy as np
from scipy import stats

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
var_row   = "Treatment"
var_col   = "Heart_Attack_Status"
alpha     = 0.05
# ──────────────────────────────────────────────────────────────────────────────

# 1. Create and display the contingency table
contingency_table = pd.crosstab(df_aspirin[var_row], df_aspirin[var_col],
                                margins=True, margins_name="Total")
print("Observed Frequencies")
print(contingency_table)


# 2. Run the test on the table without margins
contingency_table_raw = pd.crosstab(df_aspirin[var_row], df_aspirin[var_col])
chi2, p, dof, expected = stats.chi2_contingency(contingency_table_raw,
                                                 correction=False)

# 3. Show expected frequencies and validity check
expected_df = pd.DataFrame(expected,
                           index=contingency_table_raw.index,
                           columns=contingency_table_raw.columns)
print("\nExpected Frequencies")
print(expected_df.round(2))
print(f"All expected frequencies ≥ 5: {(expected >= 5).all()}")

# 4. Print test results and conclusion
print(f"\nPearson's Chi-square Test")
print(f"Variables: {var_row} (Row) vs {var_col} (Column)")
print(f"X-squared (q) = {chi2:.2f}")
print(f"df = {dof}")
print(f"p-value = {p:.2e}")
print(f"Conclusion: {'Reject H0 — significant association' if p < alpha
                     else 'Fail to reject H0 — no significant association'} (α = {alpha})")

If the code is correct, you should see the following output:
```text
Observed Frequencies
Heart_Attack_Status  Attack  No-attack  Total
Treatment                                    
Aspirin                 104      10933  11037
Placebo                 189      10845  11034
Total                   293      21778  22071

Expected Frequencies
Heart_Attack_Status  Attack  No-attack
Treatment                             
Aspirin              146.52   10890.48
Placebo              146.48   10887.52
All expected frequencies ≥ 5: True

Pearson's Chi-square Test
Variables: Treatment (Row) vs Heart_Attack_Status (Column)
X-squared (q) = 25.01
df = 1
p-value = 5.69e-07
Conclusion: Reject H0 — significant association (α = 0.05)
```

The output provides contingency tables for the observed and expected frequencies. The results of Pearson's $\chi^{2}$ test of independence are also given. In this case, the observed value of the test statistic $Q$ is $q = 24.43$, and the $p$-value is $ p_{obs} = 5.69 \times 10^{-7}$, which is quite small. Consequently, at any reasonable level of significance, we can reject the null hypothesis of independence between the variables and conclude that the results are statistically significant, and so the observed departure from the null hypothesis is quite unlikely to be due to chance alone.


## Exercise 1: Generate and Analyze Contingency Table

In the cell below, write the code to generate and analyze a contingency table for the `df_titanic` DataFrame. Test whether there is a statistically significant association between gender (`Sex`) and the passengers that survived (`Survived`) the sinking of the ocean liner.


**Code Hints:**

1. Copy-and-paste Example 1 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
var_row = "Sex"
var_col = "Survived"
alpha   = 0.05
# ──────────────────────────────────────────────────────────────────────────────

```
3. Replace every instance of `df_aspirin` with `df_titanic`.

In [ ]:
# @title Exercise 1: Generate and Analyze Contingency Table




If the code is correct, you should see the following output:
```text
Observed Frequencies
Survived   No  Yes  Total
Sex                      
female     81  233    314
male      468  109    577
Total     549  342    891

Expected Frequencies
Survived      No     Yes
Sex                     
female    193.47  120.53
male      355.53  221.47
All expected frequencies ≥ 5: True

Pearson's Chi-Squared Test of Independence
Variables: Sex (Row) vs Survived (Column)
X-squared = 263.0506
df        = 1
p-value   = 3.71e-59

Conclusion: Reject H₀ — significant association (α = 0.05)
```

The results suggest that sex was a significant factor in survival, consistent with historical accounts that women were given priority access to lifeboats during the Titanic disaster.

## **10.5 Advanced**

In this section, we discuss Fisher's exact test for analyzing contingency tables from small data sets. We also provide some commonly used Python functions for analyzing categorical data.


### **_10.5.1 Fisher's Exact Test_**

For Person's $\chi^{2}$ test to be valid, the expected frequencies $(E_{ij})$ under the null should be at least $5$, so we can assume that the distribution of $Q$ is approximately $\chi^{2}$ under the null. Occasionally, this requirement is violated (especially when the sample size is small, or the number of categories is large, or some of the categories are rare), and some of the expected frequencies become small (less than 5).

![__](https://biologicslab.co/STA1403/images/chapter_10/chapter_10B_image02B.png)

>**Table 10.8** Contingency table of diabetes status by weight status

Recall that in Sect. 2.5, we created a categorical variable called `weight.status` based on BMI values in the `Pima.tr` data set. This variable had four categories: "Underweight", "Normal", "Overweight", and "Obese". After you create this new variable, follow the above steps to perform Pearson's $\chi^{2}$ test in order to investigate the relationship between weight.status and type (disease status). You should obtain a $4\times 2$ contingency table, similar to Table 10.8, by selecting `weight.status` as the row variable and `type` as the column variable. Note that there are only two underweight women in this sample. (This seems to be a rare event in the population.)

Based on the above table, the expected frequencies $E_{1,1}=1.32$ and $E_{12}=0.68$. (The remaining expected frequencies are above 5.) Therefore, we shouldn't use the $\chi^{2}$ test, when the expected frequencies are below 5. Instead of using Pearson’s $\chi^{2}$ test (which assumes that $Q$ statistic has an approximate $\chi^{2}$ distribution), we should use **Fisher’s exact test** (which is based on the exact distribution of a test statistic that captures the deviation from the null).

Example 2 shows how to perform Fisher's exact test using Python on this data set.


## Example 2: Fisher's Exact Test

The code in the cell below shows how to perform Fisher's Exact Test when the expected frequency is too small to safely use the $Q$ statistic. In this example, we are testing whether the body mass index (`bmi`) is associated with the development of Type II diabetes (`type`).

In [ ]:
# @title Example 2: Fisher's Exact Test

import pandas as pd
from scipy import stats

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
col_bin = "bmi"
col_var = "type"
row_var = "weight_status"
labels  = ['Underweight', 'Normal', 'Overweight', 'Obese']
bins    = [0, 18.5, 25, 30, float('inf')]
alpha   = 0.05
# ──────────────────────────────────────────────────────────────────────────────

# Create a local copy
df_pima_copy = df_pima.copy()
df_pima_copy[row_var] = pd.cut(df_pima_copy[col_bin], bins=bins, labels=labels, right=False)

# 1. Observed frequencies
contingency_table = pd.crosstab(df_pima_copy[row_var], df_pima_copy[col_var],
                                margins=True, margins_name="Total")
print("Observed Frequencies")
print(contingency_table)

# 2. Fisher's Exact Test
contingency_table_raw = pd.crosstab(df_pima_copy[row_var], df_pima_copy[col_var])
odds_ratio, p = stats.fisher_exact(contingency_table_raw)

# 3. Test results and conclusion
print(f"\nFisher's Exact Test")
print(f"Variables: {row_var} (Row) vs {col_var} (Column)")
print(f"Odds Ratio = {odds_ratio:.4f}")
print(f"p-value    = {p:.4f}")
print(f"\nConclusion: {'Reject H₀ — significant association' if p < alpha
                       else 'Fail to reject H₀ — no significant association'} (α = {alpha})")

If the code is correct, you should see the following output:
```text
Observed Frequencies
type            No  Yes  Total
weight_status                 
Underweight      2    0      2
Normal          21    2     23
Overweight      35    8     43
Obese           74   58    132
Total          132   68    200

Fisher's Exact Test
Variables: weight_status (Row) vs type (Column)
Odds Ratio = 0.0000
p-value    = 0.0002

Conclusion: Reject H₀ — significant association (α = 0.05)
```

 The resulting $p$-value is $0.0002$, which is slightly lower than the $p$-value based on $\chi^{2}$ test. At $0.01$ level, we can reject the null hypothesis, which indicates that the disease status is independent from the weight status, and conclude that the relationship between the two variables is statistically significant.

## **Exercise 2: Fisher's Exact Test**

In the cell below, write the code to perform Fisher's Exact Test on the `birthwt` data set. Test whether maternal smoking (`smoke = 1`) is associated with lower birth weights (`Low`).

**Code Hints:**

1. Copy-and-paste Example 2 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
col_bin = "bwt"
col_var = "smoke"
row_var = "birthweight_status"
labels  = ["Low", "Normal"]
bins    = [0, 2500, float('inf')]
alpha   = 0.05
# ──────────────────────────────────────────────────────────────────────────────
```

3. Change every instance of `df_pima` with `df_bwt`.

In [ ]:
# @title Exercise 2: Fisher's Exact Test



If the code is correct, you should see the following output:

```text
Observed Frequencies
smoke                 0   1  Total
birthweight_status                
Low                  29  30     59
Normal               86  44    130
Total               115  74    189

Fisher's Exact Test
Variables: birthweight_status (Row) vs smoke (Column)
Odds Ratio = 0.4946
p-value    = 3.62e-02

Conclusion: Reject H₀ — significant association (α = 0.05)

```

 The resulting $p$-value is  $3.62×10^{-02}$, which is very small. At $0.01$ level, we can reject the null hypothesis, which indicates that the birthweight is independent of maternal smoking status, and conclude that the relationship between the two variables is statistically significant.

### **_10.5.2 Pearson's $\chi^{2}$ Test Using Python_**

We can also use Python to perform Pearson's $\chi^{2}$ on categorical data sets. Example 3 shows how to use Python to perform Pearson's $\chi^{2}$ Test of Independence on the the Hair Eye Color dataset. This dataset includes hair and eye color and sex for 592 statistics students at the University of Delaware reported by Snee (1974).

## Example 3: Pearson's $\chi^{2}$ Test of Independence

The code in the cell below performs Pearson's $\chi^{2}$ Test of Independence on the Hair Eye Color dataset. This dataset includes hair and eye color and sex for 592 statistics students at the University of Delaware reported by Snee (1974). The first column shows different hair colors (Black, Brown, Red, Blond), the second column shows different eye colors (Brown, Blue, Hazel, Green), and the third column shows genders (Male, Female) of students. For each row, the last column shows the number of students with a specific hair color, eye color, and gender.

In [ ]:
# @title Example 3: Pearson's χ² Test of Independence (Hair & Eye Color)

import pandas as pd
import numpy as np
from scipy import stats

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
var_row  = "Hair"
var_col  = "Eye"
freq_col = "Freq"
row_order = ['Black', 'Brown', 'Red', 'Blond']
col_order = ['Brown', 'Blue', 'Hazel', 'Green']
alpha    = 0.05
# ──────────────────────────────────────────────────────────────────────────────

# 0. Aggregate frequencies across Sex (collapse to Hair × Eye only)
df_agg = df_hair_eye.groupby([var_row, var_col])[freq_col].sum().reset_index()

# 1. Build and display observed contingency table
contingency_table_raw =  \
    df_agg.pivot_table(index=var_row, columns=var_col,
        values=freq_col,
        aggfunc='sum').loc[row_order, col_order]
contingency_table = contingency_table_raw.copy()
contingency_table["Total"] = contingency_table_raw.sum(axis=1)
contingency_table.loc["Total"] = contingency_table.sum()

print("Observed Frequencies")
print(contingency_table)

# 2. Chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(contingency_table_raw,
                                                 correction=False)

# 3. Expected frequencies and validity check
expected_df = pd.DataFrame(expected,
                           index=contingency_table_raw.index,
                           columns=contingency_table_raw.columns)
print("\nExpected Frequencies")
print(expected_df.round(2))
print(f"All expected frequencies ≥ 5: {(expected >= 5).all()}")

# 4. Results and conclusion
print(f"\nPearson's Chi-Squared Test of Independence")
print(f"Variables: {var_row} (Row) vs {var_col} (Column)")
print(f"X-squared = {chi2:.4f}")
print(f"df        = {dof}")
print(f"p-value   = {p:.2e}")
print(f"\nConclusion: {'Reject H₀ — significant association' if p < alpha
                       else 'Fail to reject H₀ — no significant association'} (α = {alpha})")

If the code is correct, you should see the following output:
```text
Observed Frequencies
Eye    Brown  Blue  Hazel  Green  Total
Hair                                   
Black     68    20     15      5    108
Brown    119    84     54     29    286
Red       26    17     14     14     71
Blond      7    94     10     16    127
Total    220   215     93     64    592

Expected Frequencies
Eye     Brown    Blue  Hazel  Green
Hair                               
Black   40.14   39.22  16.97  11.68
Brown  106.28  103.87  44.93  30.92
Red     26.39   25.79  11.15   7.68
Blond   47.20   46.12  19.95  13.73
All expected frequencies ≥ 5: True

Pearson's Chi-Squared Test of Independence
Variables: Hair (Row) vs Eye (Column)
X-squared = 138.2898
df        = 9
p-value   = 2.33e-25

Conclusion: Reject H₀ — significant association (α = 0.05)

```

The contingency table shows the distribution of hair and eye color among 592 statistics students at the University of Delaware. Visual inspection of the observed frequencies suggests some notable patterns — for example, brown-haired students tend to have brown eyes ($119$ observed vs. $106.28$ expected), while blond-haired students are strongly associated with blue eyes ($94$ observed vs. $46.12$ expected). Red-haired students appear roughly evenly distributed across eye colors, though the sample size is small.

All expected frequencies exceed $5$, confirming that Pearson's χ² test is valid. The test yields X-squared = $138.29$ with $9$ degrees of freedom and a $p$-value of $2.33×10^{-25}$, which is extremely small. We therefore reject $H_0$ and conclude that there is a highly significant association between hair color and eye color ($α = 0.05$). This is consistent with the well-known biological relationship between pigmentation genes that influence both hair and eye color simultaneously.

## **Exercise 3: Pearson's $\chi^{2}$ Test of Independence**

In the cell below, write the code to perform Pearson's $\chi^{2}$ Test of Independence on the `UCBAdmissions` dataset. This dataset records admissions decisions at UC Berkeley by department and gender.

**Code Hints:**

1. Copy-and-paste Example 3 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
var_row  = "Dept"
var_col  = "Admit"
freq_col = "Freq"
row_order = ['A', 'B', 'C', 'D', 'E', 'F']
col_order = ['Admitted', 'Rejected']
alpha    = 0.05
# ──────────────────────────────────────────────────────────────────────────────
```

3. Change every instance of `df_hair_eye` to `df_ucb`.

In [ ]:
# @title Exercise 3: Pearson's χ² Test of Independence (Hair & Eye Color)



If the code is correct, you should see the following output:
```text
Observed Frequencies
Admit  Admitted  Rejected  Total
Dept                            
A           601       332    933
B           370       138    508
C           322       347    669
D           269       579    848
E           147       480    627
F            46       668    714
Total      1755      2544   4299

Expected Frequencies
Admit  Admitted  Rejected
Dept                     
A        380.88    552.12
B        207.38    300.62
C        273.11    395.89
D        346.18    501.82
E        255.96    371.04
F        291.48    422.52
All expected frequencies ≥ 5: True

Pearson's Chi-Squared Test of Independence
Variables: Dept (Row) vs Admit (Column)
X-squared = 902.0592
df        = 5
p-value   = 9.54e-193

Conclusion: Reject H₀ — significant association (α = 0.05)
```

The contingency table shows the number of admitted and rejected applicants across six departments ($A–F$) at UC Berkeley. Admission rates vary considerably by department — Department $A$ admitted $601$ out of $933$ applicants ($64$%), while Department $F$ admitted only $46$ out of $714$ ($6$%), suggesting substantial differences in selectivity across departments.

All expected frequencies exceed $5$, confirming that Pearson's χ² test is valid here. The test yields $X$-squared = $902.06$ with $5$ degrees of freedom and a $p$-value of $9.54×10^{-193}$, which is extraordinarily small. We therefore reject $H_0$ and conclude that there is a highly significant association between department and admission outcome ($α = 0.05$).

It is worth noting that this dataset is historically famous for illustrating **Simpson's Paradox**. When admissions are aggregated across all departments, the data appear to show a bias against female applicants. However, when broken down by department, the pattern reverses — women tended to apply to more competitive departments ($C$, $D$, $E$, $F$) with lower overall admission rates, while men applied in greater numbers to departments with higher admission rates ($A$, $B$). This is a cautionary example of how aggregated statistics can be misleading without examining the underlying structure of the data.

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/STA1403/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()